# Scraper Dataset Hoaks vs Non-Hoaks (Refactor v12)

Notebook ini disiapkan untuk menghasilkan dataset scraping yang kompatibel dengan `Deteksi_Hoax_NER_Optimized.ipynb` dan `scripts/build_dataset.py`.

## Validasi Inventaris (Snapshot: 2026-03-05)

| Path | Status | Catatan |
|---|---|---|
| `Berita_Hoax/Scraper_Dataset_Hoaks_NonHoaks.ipynb` | FOUND | Notebook scraping utama |
| `Berita_Hoax/Deteksi_Hoax_NER_Optimized.ipynb` | FOUND | Notebook training/inference |
| `Berita_Hoax/scrape_output/` | FOUND | CSV, cache sitemap, progress, sqlite, log |
| `Berita_Hoax/scripts/build_dataset.py` | FOUND | Kontrak kolom downstream |
| `Berita_Hoax/data/processed/summary.json` | FOUND | Ringkasan split data processed |
| `Berita_Hoax/data/processed/leakage_audit.json` | FOUND | Audit leakage |

**MISSING:** -

## Tujuan dan Sumber Data

- Sumber non-hoaks (label `0`): CNN, Detik, Kompas.
- Sumber hoaks (label `1`): TurnBackHoax (artikel klarifikasi/klaim hoaks).
- `Narasi`: isi naratif utama berita.
- `Clean Narasi`: versi pembersihan teks untuk pemodelan.
- `summary`: ringkasan ekstraktif ringan.

## Etika Scraping

- Patuh `robots.txt` (fail-closed, tanpa bypass proteksi).
- Rate limiting + throttle per domain.
- User-Agent akademik.
- Caching sitemap agar tidak melakukan request berulang berlebihan.
- Tidak menyimpan data sensitif.

## Flow

`discovery -> fetch -> parse -> clean -> summary -> save incremental -> QA`

## Skema Output CSV

Kolom final wajib:
1. `url`
2. `judul`
3. `tanggal` (ISO `YYYY-MM-DD`)
4. `isi_berita`
5. `Narasi`
6. `Clean Narasi`
7. `hoax` (`0/1`)
8. `summary`
9. `source`

## 1) Install dependensi (jalankan sekali)
Jika environment sudah siap, cell ini bisa dilewati.

In [ ]:
%pip -q install requests beautifulsoup4 lxml pandas tqdm python-dateutil dateparser trafilatura protego sastrawi

## 2) Import & konfigurasi

In [ ]:
from __future__ import annotations

import os
import re
import json
import time
import math
import csv
import hashlib
import random
import logging
import sqlite3
import threading
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, Iterable, Iterator, List, Optional, Tuple

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import pandas as pd

from dateutil import parser as dateutil_parser
import dateparser
from trafilatura import extract as trafilatura_extract

try:
    from protego import Protego
except Exception:
    Protego = None

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory


def _versi(mod, fallback: str = "n/a") -> str:
    return getattr(mod, "__version__", fallback)

print("Versi library penting:")
print({
    "python": os.sys.version.split()[0],
    "pandas": _versi(pd),
    "requests": _versi(requests),
    "beautifulsoup4": _versi(__import__("bs4")),
    "dateparser": _versi(dateparser),
    "trafilatura": _versi(__import__("trafilatura")),
    "protego": _versi(__import__("protego")) if Protego is not None else "not-installed",
})

### 2.1) Struktur folder (disarankan)
Jalankan notebook dari root proyek `Berita_Hoax/`.
```
Berita_Hoax/
  Scraper_Dataset_Hoaks_NonHoaks.ipynb
  Deteksi_Hoax_NER_Optimized.ipynb
  scripts/build_dataset.py
  scrape_output/
    logs/
    cache/
    progress_scrape.json
    progress_done_urls.sqlite
    failed_urls.jsonl
    data_hoaks_turnbackhoaks.csv
    data_nonhoaks_cnn.csv
    data_nonhoaks_detik.csv
    data_nonhoaks_kompas.csv
```
Semua path notebook menggunakan `pathlib.Path` (tanpa path absolut Windows).

In [ ]:
def deteksi_root_proyek() -> Path:
    cwd = Path.cwd().resolve()
    kandidat = [cwd, *cwd.parents]
    for p in kandidat:
        if (p / "scripts" / "build_dataset.py").exists() and (p / "Deteksi_Hoax_NER_Optimized.ipynb").exists():
            return p
    return cwd


@dataclass
class Konfigurasi:
    # Root proyek
    project_root: Path = field(default_factory=deteksi_root_proyek)

    # Rentang tanggal (wajib)
    start_date: str = "2020-01-01"  # YYYY-MM-DD
    end_date: Optional[str] = None   # None = hari ini saat runtime

    # Output
    output_subdir: str = "scrape_output"
    gabungkan_output: bool = True

    # Mode
    mode_smoke_test: bool = False
    smoke_limit_url_discovery: int = 200
    smoke_limit_scrape_per_sumber: int = 50

    # HTTP
    user_agent: str = "BeritaHoaxDatasetBot/1.0 (contact: fajarmuhairi@student.uir.ac.id; purpose: academic research)"
    timeout_connect: int = 10
    timeout_read: int = 30
    max_retries: int = 4
    backoff_factor: float = 0.8

    # Robots
    robots_timeout_detik: int = 30
    robots_timeout_lain: int = 20

    # Sitemap
    batas_sitemap_rekursif: Optional[int] = None

    # Kinerja
    max_workers_total: Optional[int] = None
    max_koneksi_per_domain: int = 6
    min_jeda_per_domain_detik: float = 0.25

    # Parsing tanggal
    zona_waktu_default: str = "Asia/Jakarta"

    # Progress/resume
    progress_path: Path = field(init=False)
    failed_path: Path = field(init=False)
    done_db_path: Path = field(init=False)

    # TBH paging
    tbh_max_pages: Optional[int] = None

    # Turunan path
    output_dir: Path = field(init=False)
    logs_dir: Path = field(init=False)
    cache_dir: Path = field(init=False)

    def __post_init__(self):
        self.project_root = self.project_root.resolve()
        if self.end_date is None:
            self.end_date = pd.Timestamp.now(tz=self.zona_waktu_default).date().isoformat()

        self.output_dir = (self.project_root / self.output_subdir).resolve()
        self.logs_dir = self.output_dir / "logs"
        self.cache_dir = self.output_dir / "cache"

        self.progress_path = self.output_dir / "progress_scrape.json"
        self.failed_path = self.output_dir / "failed_urls.jsonl"
        self.done_db_path = self.output_dir / "progress_done_urls.sqlite"

        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.logs_dir.mkdir(parents=True, exist_ok=True)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

        cpu_logical = os.cpu_count() or 4
        if self.max_workers_total is None:
            self.max_workers_total = min(24, max(6, cpu_logical * 2))


cfg = Konfigurasi()

print("project_root:", cfg.project_root)
print("output_dir:", cfg.output_dir)
print("rentang_tanggal:", cfg.start_date, "s.d.", cfg.end_date)
print("mode_smoke_test:", cfg.mode_smoke_test)
print("max_workers_total:", cfg.max_workers_total)

## 2.2) Validasi inventaris dan scan awal `scrape_output/` (WAJIB PERTAMA)
Cell ini menampilkan:
- status FOUND/MISSING file kunci
- daftar file utama di `scrape_output/` beserta ukuran dan waktu modifikasi
- ringkasan peran artefak (cache sitemap, progress, sqlite done-url, failed urls, logs)

In [ ]:
def _ukuran_human(n: int) -> str:
    step = 1024.0
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for u in units:
        if x < step or u == units[-1]:
            return f"{x:.2f} {u}"
        x /= step
    return f"{n} B"


def _waktu(ts: float) -> str:
    return pd.to_datetime(ts, unit="s").strftime("%Y-%m-%d %H:%M:%S")


kunci = [
    cfg.project_root / "Scraper_Dataset_Hoaks_NonHoaks.ipynb",
    cfg.project_root / "Deteksi_Hoax_NER_Optimized.ipynb",
    cfg.project_root / "scrape_output",
    cfg.project_root / "scripts" / "build_dataset.py",
    cfg.project_root / "data" / "processed" / "summary.json",
    cfg.project_root / "data" / "processed" / "leakage_audit.json",
]

inv_rows = []
for p in kunci:
    inv_rows.append({
        "path": str(p.relative_to(cfg.project_root)),
        "status": "FOUND" if p.exists() else f"MISSING: {p}",
        "is_dir": p.is_dir() if p.exists() else False,
    })

display(pd.DataFrame(inv_rows))

if cfg.output_dir.exists():
    files = [x for x in cfg.output_dir.rglob("*") if x.is_file()]
    rows = []
    for p in files:
        rel = p.relative_to(cfg.output_dir).as_posix()
        role = ""
        if rel.startswith("cache/sitemap_"):
            role = "cache sitemap XML"
        elif rel.startswith("cache/audit_urls_"):
            role = "cache QA dedup URL"
        elif rel == "progress_done_urls.sqlite":
            role = "sqlite done-url resume"
        elif rel == "progress_scrape.json":
            role = "checkpoint discovery/scrape"
        elif rel == "failed_urls.jsonl":
            role = "rekam URL gagal"
        elif rel == "logs/scrape.log":
            role = "log scraping"
        elif rel.endswith(".csv"):
            role = "dataset CSV output"

        rows.append({
            "file": rel,
            "size": _ukuran_human(p.stat().st_size),
            "last_modified": _waktu(p.stat().st_mtime),
            "role": role,
        })

    df_files = pd.DataFrame(rows).sort_values("file")
    print("Total file di scrape_output:", len(df_files))
    display(df_files.head(60))

    kategori = Counter()
    for rel in df_files["file"].tolist():
        if rel.startswith("cache/"):
            kategori["cache"] += 1
        elif rel.startswith("logs/"):
            kategori["logs"] += 1
        elif rel.endswith(".csv"):
            kategori["csv"] += 1
        elif rel.endswith(".json"):
            kategori["json"] += 1
        elif rel.endswith(".jsonl"):
            kategori["jsonl"] += 1
        elif rel.endswith(".sqlite"):
            kategori["sqlite"] += 1
        else:
            kategori["lainnya"] += 1
    print("Ringkasan kategori:", dict(kategori))
else:
    print(f"MISSING: {cfg.output_dir}")

## 3) Logging

In [ ]:
logger = logging.getLogger("scraper_v12")
logger.setLevel(logging.INFO)
logger.propagate = False

for h in list(logger.handlers):
    logger.removeHandler(h)

_fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

fh = logging.FileHandler(cfg.logs_dir / "scrape.log", encoding="utf-8")
fh.setFormatter(_fmt)
fh.setLevel(logging.INFO)
logger.addHandler(fh)

sh = logging.StreamHandler()
sh.setFormatter(_fmt)
sh.setLevel(logging.INFO)
logger.addHandler(sh)

logger.info("Mulai v12. output_dir=%s", cfg.output_dir)

## 4) Progress (resume) + Done-URL store (SQLite)
Versi ini menyimpan URL sukses pada tabel `done_success_urls` agar rerun bersifat incremental tanpa menduplikasi data.

In [ ]:
def _tulis_json_atomik(path: Path, data: Dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def muat_progress(path: Path) -> Dict:
    if path.exists():
        try:
            teks = path.read_text(encoding="utf-8").strip()
            if teks:
                return json.loads(teks)
            raise json.JSONDecodeError("Empty progress file", "", 0)
        except Exception as e:
            backup = path.with_suffix(path.suffix + f".broken_{int(time.time())}")
            try:
                os.replace(path, backup)
            except Exception:
                pass
            logger.warning("Progress rusak/invalid. Di-backup ke %s. Alasan: %s", backup, e)

    return {
        "sumber": {},
        "restricted_urls": [],
        "gagal_scrape": [],
        "meta": {"created_at": time.time(), "version": 12},
    }


progress = muat_progress(cfg.progress_path)


class DoneURLStore:
    def __init__(self, db_path: Path):
        self.db_path = db_path
        self._init_db()

    def _conn(self):
        con = sqlite3.connect(self.db_path)
        con.execute("PRAGMA journal_mode=WAL;")
        con.execute("PRAGMA synchronous=NORMAL;")
        return con

    def _init_db(self):
        con = self._conn()
        try:
            con.execute(
                """
                CREATE TABLE IF NOT EXISTS done_success_urls (
                    source TEXT NOT NULL,
                    url TEXT NOT NULL,
                    PRIMARY KEY (source, url)
                )
                """
            )
            con.commit()
        finally:
            con.close()

    def add_many(self, source: str, urls: List[str]):
        if not urls:
            return
        con = self._conn()
        try:
            con.executemany(
                "INSERT OR IGNORE INTO done_success_urls(source, url) VALUES (?, ?)",
                [(source, u) for u in urls if u],
            )
            con.commit()
        finally:
            con.close()

    def filter_not_done(self, source: str, urls: List[str]) -> List[str]:
        if not urls:
            return []
        con = self._conn()
        try:
            not_done = []
            step = 900
            for i in range(0, len(urls), step):
                batch = urls[i : i + step]
                q_marks = ",".join(["?"] * len(batch))
                rows = con.execute(
                    f"SELECT url FROM done_success_urls WHERE source=? AND url IN ({q_marks})",
                    [source] + batch,
                ).fetchall()
                done_set = {r[0] for r in rows}
                not_done.extend([u for u in batch if u not in done_set])
            return not_done
        finally:
            con.close()

    def seed_from_csv(self, source: str, csv_path: Path):
        if not csv_path.exists():
            return
        try:
            for chunk in pd.read_csv(csv_path, usecols=["url"], chunksize=200000, dtype=str):
                urls = chunk["url"].fillna("").astype(str).str.strip().tolist()
                self.add_many(source, [u for u in urls if u])
        except Exception as e:
            logger.warning("Seed done_success_urls gagal (%s): %s", csv_path, e)

    def count_by_source(self) -> Dict[str, int]:
        con = self._conn()
        try:
            rows = con.execute(
                "SELECT source, COUNT(*) FROM done_success_urls GROUP BY source ORDER BY COUNT(*) DESC"
            ).fetchall()
            return {src: int(cnt) for src, cnt in rows}
        finally:
            con.close()


done_store = DoneURLStore(cfg.done_db_path)


def seed_done_store_dari_csv():
    mapping = {
        "turnbackhoaks": cfg.output_dir / "data_hoaks_turnbackhoaks.csv",
        "cnn": cfg.output_dir / "data_nonhoaks_cnn.csv",
        "detik": cfg.output_dir / "data_nonhoaks_detik.csv",
        "kompas": cfg.output_dir / "data_nonhoaks_kompas.csv",
    }
    for sumber, p in mapping.items():
        done_store.seed_from_csv(sumber, p)


def mark_meta_progress(progress: Dict, sumber: str, **kwargs):
    progress.setdefault("sumber", {})
    progress["sumber"].setdefault(sumber, {})
    progress["sumber"][sumber].setdefault("meta", {})
    progress["sumber"][sumber]["meta"].update(kwargs)
    _tulis_json_atomik(cfg.progress_path, progress)

## 5) Robots.txt (patuh)
Implementasi berlapis:
- Jika `protego` ada: pakai Protego (lebih lengkap).
- Jika tidak: fallback ke `urllib.robotparser`.

In [ ]:
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser


def _can_fetch_dynamic(rob, user_agent: str, url: str) -> bool:
    fn = getattr(rob, "can_fetch", None)
    if fn is None:
        return False
    for args in [(user_agent, url), (url, user_agent), (url,)]:
        try:
            return bool(fn(*args))
        except TypeError:
            continue
        except Exception:
            return False
    return False


class RobotsManager:
    def __init__(self):
        self._lock = threading.Lock()
        self._robots: Dict[str, object] = {}
        self._loaded: Dict[str, Tuple[bool, str]] = {}
        self._sitemaps: Dict[str, List[str]] = {}

    def _timeout_for(self, domain: str) -> int:
        if "detik.com" in domain:
            return cfg.robots_timeout_detik
        return cfg.robots_timeout_lain

    def load(self, base_url: str, session: Optional[requests.Session] = None) -> Tuple[bool, str]:
        parsed = urlparse(base_url)
        domain = parsed.netloc
        scheme = parsed.scheme or "https"

        with self._lock:
            if domain in self._loaded:
                return self._loaded[domain]

        robots_url = f"{scheme}://{domain}/robots.txt"
        try:
            sess = session if session is not None else requests.Session()
            r = sess.get(
                robots_url,
                timeout=(cfg.timeout_connect, self._timeout_for(domain)),
                headers={"User-Agent": cfg.user_agent},
            )
            if r.status_code >= 400:
                raise RuntimeError(f"status={r.status_code}")
            teks = r.text or ""

            sitemaps = []
            for ln in teks.splitlines():
                if ln.lower().startswith("sitemap:"):
                    sm = ln.split(":", 1)[1].strip()
                    if sm.startswith("http"):
                        sitemaps.append(sm)

            with self._lock:
                self._sitemaps[domain] = list(dict.fromkeys(sitemaps))

            if Protego is not None:
                try:
                    rob = Protego.parse(teks)
                except Exception:
                    rp = RobotFileParser()
                    rp.parse(teks.splitlines())
                    rob = rp
            else:
                rp = RobotFileParser()
                rp.parse(teks.splitlines())
                rob = rp

            with self._lock:
                self._robots[domain] = rob
                self._loaded[domain] = (True, "")
            return True, ""
        except Exception as e:
            with self._lock:
                self._robots[domain] = None
                self._loaded[domain] = (False, str(e))
            return False, str(e)

    def get_sitemaps(self, base_url: str) -> List[str]:
        domain = urlparse(base_url).netloc
        with self._lock:
            return self._sitemaps.get(domain, [])

    def can_fetch(self, url: str) -> bool:
        parsed = urlparse(url)
        domain = parsed.netloc
        if not domain:
            return False

        if domain not in self._robots:
            try:
                base = f"{parsed.scheme or 'https'}://{domain}"
                self.load(base)
            except Exception:
                return False

        rob = self._robots.get(domain)
        if rob is None:
            return False

        return _can_fetch_dynamic(rob, cfg.user_agent, url)

    def boleh_fetch(self, url: str) -> bool:
        return self.can_fetch(url)

    def info(self) -> pd.DataFrame:
        rows = []
        with self._lock:
            for domain, (ok, note) in self._loaded.items():
                rows.append({"domain": f"https://{domain}", "robots_loaded": ok, "note": note})
        return pd.DataFrame(rows)


robots = RobotsManager()

## 6) HTTP client + retry/backoff + throttle per domain

In [ ]:
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

_thread_local = threading.local()
_failed_lock = threading.Lock()


class DomainLimiter:
    def __init__(self, max_concurrent: int, min_delay: float):
        self.max_concurrent = max_concurrent
        self.min_delay = min_delay
        self._sem: Dict[str, threading.Semaphore] = {}
        self._last_ts: Dict[str, float] = {}
        self._lock = threading.Lock()

    def _get_sem(self, domain: str) -> threading.Semaphore:
        with self._lock:
            if domain not in self._sem:
                self._sem[domain] = threading.Semaphore(self.max_concurrent)
            return self._sem[domain]

    def acquire(self, domain: str):
        sem = self._get_sem(domain)
        sem.acquire()
        with self._lock:
            last = self._last_ts.get(domain, 0.0)
            now = time.time()
            delta = now - last
            if delta < self.min_delay:
                time.sleep(self.min_delay - delta)
            self._last_ts[domain] = time.time()

    def release(self, domain: str):
        sem = self._get_sem(domain)
        sem.release()


limiter = DomainLimiter(cfg.max_koneksi_per_domain, cfg.min_jeda_per_domain_detik)


def get_session() -> requests.Session:
    s = getattr(_thread_local, "session", None)
    if s is not None:
        return s

    s = requests.Session()
    retry = Retry(
        total=cfg.max_retries,
        connect=cfg.max_retries,
        read=cfg.max_retries,
        backoff_factor=cfg.backoff_factor,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET", "HEAD"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    s.headers.update({"User-Agent": cfg.user_agent})
    _thread_local.session = s
    return s


def http_get(url: str, *, timeout: Optional[Tuple[int, int]] = None) -> requests.Response:
    domain = urlparse(url).netloc
    limiter.acquire(domain)
    try:
        s = get_session()
        t = timeout if timeout is not None else (cfg.timeout_connect, cfg.timeout_read)
        return s.get(url, timeout=t)
    finally:
        limiter.release(domain)


def catat_failed(url: str, sumber: str, tahap: str, alasan: str):
    row = {"ts": time.time(), "url": url, "source": sumber, "stage": tahap, "reason": alasan}
    line = json.dumps(row, ensure_ascii=False)
    with _failed_lock:
        with cfg.failed_path.open("a", encoding="utf-8") as f:
            f.write(line + "\n")

## 7) Util: parsing tanggal (berlapis + validasi)

In [ ]:
BULAN_ID = {
    "jan": 1, "januari": 1,
    "feb": 2, "februari": 2,
    "mar": 3, "maret": 3,
    "apr": 4, "april": 4,
    "mei": 5,
    "jun": 6, "juni": 6,
    "jul": 7, "juli": 7,
    "agu": 8, "agustus": 8,
    "sep": 9, "sept": 9, "september": 9,
    "okt": 10, "oktober": 10,
    "nov": 11, "november": 11,
    "des": 12, "desember": 12,
}


def _tahun_masuk_akal(y: int) -> bool:
    tahun_ini = pd.Timestamp.now(tz=cfg.zona_waktu_default).year
    return 2000 <= y <= (tahun_ini + 1)


def _iso_valid(y: int, mo: int, d: int) -> Optional[str]:
    if not (_tahun_masuk_akal(y) and 1 <= mo <= 12 and 1 <= d <= 31):
        return None
    try:
        return pd.Timestamp(year=y, month=mo, day=d).date().isoformat()
    except Exception:
        return None


def _parse_yearfirst_regex(teks: str) -> Optional[str]:
    m = re.search(r"\b(20\d{2})[-/.](\d{1,2})[-/.](\d{1,2})\b", teks)
    if not m:
        return None
    y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
    return _iso_valid(y, mo, d)


def _parse_dmy_regex(teks: str) -> Optional[str]:
    m = re.search(r"\b(\d{1,2})[-/](\d{1,2})[-/](20\d{2})\b", teks)
    if not m:
        return None
    d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
    return _iso_valid(y, mo, d)


def _parse_bulan_id(teks: str) -> Optional[str]:
    m = re.search(r"\b(\d{1,2})\s+([A-Za-z]+)\s+(20\d{2})\b", teks)
    if not m:
        return None
    d = int(m.group(1))
    bulan = m.group(2).strip().lower()
    y = int(m.group(3))
    mo = BULAN_ID.get(bulan)
    if mo is None:
        return None
    return _iso_valid(y, mo, d)


def parse_tanggal_aman(raw: str) -> Optional[str]:
    if not raw:
        return None

    teks = str(raw).strip()
    teks = re.sub(r"\s+", " ", teks)
    teks = re.sub(r"^Kompas\.com\s*-\s*", "", teks, flags=re.I)
    teks = re.sub(r"\b(WIB|WITA|WIT|UTC|GMT)\b", "", teks, flags=re.I).strip()

    # Prioritaskan year-first supaya 2026-02-12 tidak kebaca 2026-12-02.
    for fn in (_parse_yearfirst_regex, _parse_dmy_regex, _parse_bulan_id):
        iso = fn(teks)
        if iso:
            return iso

    try:
        dt_obj = pd.to_datetime(teks, errors="raise", dayfirst=False, yearfirst=True)
        if _tahun_masuk_akal(dt_obj.year):
            return dt_obj.date().isoformat()
    except Exception:
        pass

    try:
        dt_obj = dateparser.parse(
            teks,
            languages=["id", "en"],
            settings={
                "DATE_ORDER": "DMY",
                "PREFER_DATES_FROM": "past",
                "STRICT_PARSING": False,
            },
        )
        if dt_obj and _tahun_masuk_akal(dt_obj.year):
            return dt_obj.date().isoformat()
    except Exception:
        pass

    try:
        dt_obj = dateutil_parser.parse(teks, fuzzy=True, dayfirst=True, yearfirst=False)
        if dt_obj and _tahun_masuk_akal(dt_obj.year):
            return dt_obj.date().isoformat()
    except Exception:
        pass

    return None


def tanggal_dari_url(url: str) -> Optional[str]:
    # pola /YYYY/MM/DD/
    m = re.search(r"/(20\d{2})/(\d{2})/(\d{2})(?:/|$)", url)
    if m:
        return _iso_valid(int(m.group(1)), int(m.group(2)), int(m.group(3)))

    # pola YYYY-MM-DD
    m = re.search(r"(20\d{2})-(\d{2})-(\d{2})", url)
    if m:
        return _iso_valid(int(m.group(1)), int(m.group(2)), int(m.group(3)))

    # pola compact YYYYMMDD (mis. CNN: 20260219125548)
    for m in re.finditer(r"(20\d{2})(\d{2})(\d{2})", url):
        iso = _iso_valid(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        if iso:
            return iso

    return None


def ekstrak_tanggal_berlapis(soup: BeautifulSoup, url: str) -> Tuple[Optional[str], str]:
    # 1) JSON-LD
    for sc in soup.find_all("script", attrs={"type": "application/ld+json"}):
        teks = (sc.get_text() or "").strip()
        if not teks:
            continue
        try:
            data = json.loads(teks)
        except Exception:
            continue
        nodes: List[Dict] = []
        if isinstance(data, dict):
            if "@graph" in data and isinstance(data["@graph"], list):
                nodes.extend([x for x in data["@graph"] if isinstance(x, dict)])
            else:
                nodes.append(data)
        elif isinstance(data, list):
            nodes.extend([x for x in data if isinstance(x, dict)])
        for node in nodes:
            for k in ("datePublished", "dateModified", "uploadDate"):
                if k in node:
                    iso = parse_tanggal_aman(str(node.get(k)))
                    if iso:
                        return iso, f"jsonld:{k}"

    # 2) meta tags
    meta_kandidat = [
        ("property", "article:published_time"),
        ("property", "article:modified_time"),
        ("property", "og:published_time"),
        ("property", "og:updated_time"),
        ("name", "publish_date"),
        ("name", "publishdate"),
        ("name", "date"),
        ("name", "parsely-pub-date"),
        ("name", "dc.date.issued"),
        ("name", "DC.date.issued"),
        ("name", "timestamp"),
    ]
    for attr, key in meta_kandidat:
        tag = soup.find("meta", attrs={attr: key})
        if tag and tag.get("content"):
            iso = parse_tanggal_aman(tag["content"])
            if iso:
                return iso, f"meta:{attr}={key}"

    # 3) HTML selectors
    selector_kandidat = [
        "div.detail__date",
        "time[datetime]",
        "div.read__time",
        "span.read__time",
        "div.article__date",
        "div.date",
        "span.date",
    ]
    for sel in selector_kandidat:
        el = soup.select_one(sel)
        if not el:
            continue
        raw = el.get("datetime") or el.get_text(" ", strip=True)
        iso = parse_tanggal_aman(raw)
        if iso:
            return iso, f"html:{sel}"

    # 4) fallback URL
    iso = tanggal_dari_url(url)
    if iso:
        return iso, "urlpattern"
    return None, "none"


def dalam_rentang_2020(iso: str) -> bool:
    if not iso:
        return False
    try:
        d = pd.to_datetime(iso, errors="raise").date()
    except Exception:
        return False
    start = pd.to_datetime(cfg.start_date).date()
    end = pd.to_datetime(cfg.end_date).date() if cfg.end_date else pd.Timestamp.today().date()
    return start <= d <= end

## 8) Util: ekstraksi teks artikel
Pendekatan:
- HTML selector spesifik (lebih deterministik)
- fallback ke trafilatura (menghapus noise nav/iklan)

In [ ]:
def _teks_bersih(s: str) -> str:
    s = re.sub(r"\s+", " ", s or "").strip()
    return s

def ekstrak_judul(soup: BeautifulSoup) -> str:
    h1 = soup.find("h1")
    if h1:
        return _teks_bersih(h1.get_text(" ", strip=True))
    if soup.title:
        return _teks_bersih(soup.title.get_text(" ", strip=True))
    return ""

def ekstrak_isi_dengan_selector(soup: BeautifulSoup, selector_list: List[str]) -> str:
    for sel in selector_list:
        el = soup.select_one(sel)
        if not el:
            continue
        # ambil paragraf
        ps = el.find_all(["p", "div"])
        teks = " ".join([p.get_text(" ", strip=True) for p in ps]) if ps else el.get_text(" ", strip=True)
        teks = _teks_bersih(teks)
        if len(teks) >= 200:
            return teks
    return ""

def ekstrak_isi_trafilatura(html: str, url: str) -> str:
    try:
        teks = trafilatura_extract(html, url=url, include_comments=False, include_tables=False)
        return _teks_bersih(teks or "")
    except Exception:
        return ""

### 8.1) Parser per sumber

In [ ]:
def parse_cnn(html: str, url: str) -> Dict:
    soup = BeautifulSoup(html, "lxml")
    judul = ekstrak_judul(soup)
    tanggal_iso, tanggal_src = ekstrak_tanggal_berlapis(soup, url)
    isi = ekstrak_isi_dengan_selector(
        soup,
        selector_list=[
            "div.detail_text",
            "div.detail__body",
            "div.content_detail",
            "article",
        ],
    )
    if not isi:
        isi = ekstrak_isi_trafilatura(html, url)
    return {"judul": judul, "tanggal": tanggal_iso, "tanggal_src": tanggal_src, "isi_berita": isi}

def parse_detik(html: str, url: str) -> Dict:
    soup = BeautifulSoup(html, "lxml")
    judul = ekstrak_judul(soup)
    tanggal_iso, tanggal_src = ekstrak_tanggal_berlapis(soup, url)
    isi = ekstrak_isi_dengan_selector(
        soup,
        selector_list=[
            "div.detail__body-text",
            "div.detail__body",
            "article",
        ],
    )
    if not isi:
        isi = ekstrak_isi_trafilatura(html, url)
    return {"judul": judul, "tanggal": tanggal_iso, "tanggal_src": tanggal_src, "isi_berita": isi}

def parse_kompas(html: str, url: str) -> Dict:
    soup = BeautifulSoup(html, "lxml")
    judul = ekstrak_judul(soup)
    tanggal_iso, tanggal_src = ekstrak_tanggal_berlapis(soup, url)
    isi = ekstrak_isi_dengan_selector(
        soup,
        selector_list=[
            "div.read__content",
            "div.pek-text",
            "article",
        ],
    )
    if not isi:
        isi = ekstrak_isi_trafilatura(html, url)
    return {"judul": judul, "tanggal": tanggal_iso, "tanggal_src": tanggal_src, "isi_berita": isi}

def parse_turnbackhoaks(html: str, url: str) -> Dict:
    soup = BeautifulSoup(html, "lxml")
    judul = ekstrak_judul(soup)
    tanggal_iso, tanggal_src = ekstrak_tanggal_berlapis(soup, url)
    # klaim / narasi hoaks biasanya di awal konten
    # heuristik: ambil paragraf pertama yang panjang
    konten = ekstrak_isi_dengan_selector(
        soup,
        selector_list=[
            "div.entry-content",
            "article",
        ],
    )
    if not konten:
        konten = ekstrak_isi_trafilatura(html, url)
    klaim = ""
    if konten:
        # ambil 1-3 kalimat awal sebagai klaim
        klaim = ringkas_ekstraktif(konten, max_kalimat=2)
    return {"judul": judul, "tanggal": tanggal_iso, "tanggal_src": tanggal_src, "isi_berita": konten, "klaim": klaim}

## 9) Cleaning teks (Clean Narasi)
Implementasi: lowercase, hapus url/html, normalisasi repetisi, stopword removal, stemming ringan (Sastrawi).

In [ ]:
_stemmer = StemmerFactory().create_stemmer()
_stop_factory = StopWordRemoverFactory()
_stopwords = set(_stop_factory.get_stop_words())

_re_url = re.compile(r"https?://\S+|www\.\S+")
_re_nonword = re.compile(r"[^0-9a-zA-Z\s]+")
_re_repeat = re.compile(r"(.)\1{2,}")

def bersihkan_teks_id(teks: str) -> str:
    if not teks:
        return ""
    t = teks.lower()
    t = _re_url.sub(" ", t)
    t = re.sub(r"<[^>]+>", " ", t)
    t = _re_repeat.sub(r"\1\1", t)  # 'aaaa' -> 'aa'
    t = _re_nonword.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    if not t:
        return ""
    tokens = [w for w in t.split() if w not in _stopwords and len(w) > 1]
    if not tokens:
        return ""
    # stemming ringan: pada teks panjang, stemming full bisa berat, tetapi masih feasible batch kecil
    t2 = " ".join(tokens)
    try:
        t2 = _stemmer.stem(t2)
    except Exception:
        pass
    t2 = re.sub(r"\s+", " ", t2).strip()
    return t2

## 10) Summary (ringan, tanpa model berat)

In [ ]:
def ringkas_ekstraktif(teks: str, max_kalimat: int = 2) -> str:
    if not teks:
        return ""
    # pecah kalimat sederhana
    kalimat = re.split(r"(?<=[\.!\?])\s+", teks.strip())
    kalimat = [k.strip() for k in kalimat if k.strip()]
    return " ".join(kalimat[:max_kalimat])

## 11) Discovery URL
- CNN/Detik/Kompas: sitemap (XML)
- TurnBackHoaks: paging `/articles?page=N`

In [ ]:
import xml.etree.ElementTree as ET

def _base_domain(host: str) -> str:
    # sederhana: ambil 2 label terakhir (cukup untuk .com/.id)
    parts = host.split(".")
    if len(parts) <= 2:
        return host
    return ".".join(parts[-2:])

def _baca_xml_bytes(data: bytes) -> ET.Element:
    return ET.fromstring(data)

def iter_url_dari_sitemap(sitemap_url: str, batas_sitemap: Optional[int] = None) -> Iterator[Tuple[str, Optional[str]]]:
    # robots check
    if not robots.can_fetch(sitemap_url):
        raise RuntimeError(f"RESTRICTED by robots: {sitemap_url}")
    # cache file sitemap agar tidak berulang
    cache_key = hashlib.sha1(sitemap_url.encode("utf-8")).hexdigest()
    cache_path = os.path.join(cfg.output_dir, "cache", f"sitemap_{cache_key}.xml")
    data = None
    if os.path.exists(cache_path):
        try:
            data = open(cache_path, "rb").read()
        except Exception:
            data = None
    if data is None:
        r = http_get(sitemap_url)
        if r.status_code != 200 or not (r.content or b""):
            raise RuntimeError(f"Empty/invalid sitemap response ({r.status_code})")
        data = r.content
        try:
            open(cache_path, "wb").write(data)
        except Exception:
            pass

    root = _baca_xml_bytes(data)
    ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}
    # sitemap index
    sitemaps = root.findall("sm:sitemap", ns)
    if sitemaps:
        count = 0
        parent_host = urlparse(sitemap_url).netloc
        parent_base = _base_domain(parent_host)
        for sm in sitemaps:
            loc = sm.findtext("sm:loc", default="", namespaces=ns).strip()
            lastmod = sm.findtext("sm:lastmod", default="", namespaces=ns).strip() or None
            if not loc:
                continue
            child_host = urlparse(loc).netloc
            if _base_domain(child_host) != parent_base:
                # tetap patuh: skip host beda base-domain
                logger.warning("Skip cross-base-domain child sitemap (%s != %s): %s", child_host, parent_host, loc)
                continue
            yield from iter_url_dari_sitemap(loc, batas_sitemap=None)
            count += 1
            if batas_sitemap is not None and count >= batas_sitemap:
                break
        return

    # urlset
    urls = root.findall("sm:url", ns)
    for u in urls:
        loc = u.findtext("sm:loc", default="", namespaces=ns).strip()
        lastmod = u.findtext("sm:lastmod", default="", namespaces=ns).strip() or None
        if loc:
            yield loc, lastmod

def ambil_url_dari_sitemap_dengan_filter(
    sitemap_url: str,
    filter_regex: Optional[str] = None,
    batas_url: Optional[int] = None,
) -> List[str]:
    out: List[str] = []
    rx = re.compile(filter_regex) if filter_regex else None
    start = pd.to_datetime(cfg.start_date).date()
    end = pd.to_datetime(cfg.end_date).date() if cfg.end_date else pd.Timestamp.today().date()

    for loc, lastmod in iter_url_dari_sitemap(sitemap_url, batas_sitemap=cfg.batas_sitemap_rekursif):
        if rx and not rx.search(loc):
            continue
        # filter kasar via lastmod bila ada
        if lastmod:
            iso = parse_tanggal_aman(lastmod)
            if iso:
                d = pd.to_datetime(iso).date()
                if d < start:
                    continue
                if d > end:
                    continue
        out.append(loc)
        if batas_url is not None and len(out) >= batas_url:
            break
    return out

def temukan_url_cnn() -> List[str]:
    sitemap_utama = "https://www.cnnindonesia.com/sitemap.xml"
    regex = r"^https?://([a-z0-9-]+\.)?cnnindonesia\.com/(?!video/|tv/|tag/|search)"
    batas = cfg.smoke_limit_url_discovery if cfg.mode_smoke_test else None

    urls: List[str] = []
    kandidat = [sitemap_utama]
    kandidat += robots.get_sitemaps("https://www.cnnindonesia.com")
    kandidat = list(dict.fromkeys([k for k in kandidat if k]))

    for sm in kandidat:
        try:
            urls.extend(ambil_url_dari_sitemap_dengan_filter(sm, filter_regex=regex, batas_url=None))
        except Exception as e:
            logger.warning("Discovery CNN gagal pada %s: %s", sm, e)
            continue
        if batas is not None and len(urls) >= batas:
            urls = urls[:batas]
            break
    return list(dict.fromkeys(urls))

def temukan_url_detik() -> List[str]:
    sitemap = "https://www.detik.com/sitemap.xml"
    regex = r"^https?://(www\.)?detik\.com/(?!video/|foto/|tag/|search)"
    batas = cfg.smoke_limit_url_discovery if cfg.mode_smoke_test else None
    return ambil_url_dari_sitemap_dengan_filter(sitemap, filter_regex=regex, batas_url=batas)

def temukan_url_kompas() -> List[str]:
    sitemap = "https://www.kompas.com/sitemap.xml"
    regex = r"^https?://(www\.)?kompas\.com/(?!search/|komentar/|copy/)"
    batas = cfg.smoke_limit_url_discovery if cfg.mode_smoke_test else None
    return ambil_url_dari_sitemap_dengan_filter(sitemap, filter_regex=regex, batas_url=batas)

def temukan_url_turnbackhoaks() -> List[str]:
    base = "https://turnbackhoax.id/articles"
    batas_url = cfg.smoke_limit_url_discovery if cfg.mode_smoke_test else None

    urls: List[str] = []
    seen: set = set()
    halaman = int(progress.get("sumber", {}).get("turnbackhoaks", {}).get("discovery_state", {}).get("page") or 1)
    max_pages = cfg.tbh_max_pages

    while True:
        if max_pages is not None and halaman > max_pages:
            break
        list_url = base if halaman == 1 else f"{base}?page={halaman}"
        # robots check untuk halaman listing
        if not robots.can_fetch(list_url):
            raise RuntimeError(f"RESTRICTED by robots: {list_url}")
        r = http_get(list_url, timeout=(cfg.timeout_connect, cfg.timeout_read))
        if r.status_code != 200:
            logger.warning("TBH list status %s: %s", r.status_code, list_url)
            break
        soup = BeautifulSoup(r.text, "lxml")
        links = []
        for a in soup.select("a[href]"):
            href = a.get("href") or ""
            if "/articles/" in href:
                if href.startswith("/"):
                    href = "https://turnbackhoax.id" + href
                if href.startswith("http") and "turnbackhoax.id/articles/" in href:
                    links.append(href.split("#")[0])
        links = list(dict.fromkeys(links))
        if not links:
            break
        sebelum = len(urls)
        for u in links:
            if u not in seen:
                seen.add(u)
                urls.append(u)
        if len(urls) == sebelum:
            # tidak ada URL baru
            break

        progress["sumber"].setdefault("turnbackhoaks", {})
        progress["sumber"]["turnbackhoaks"]["discovery_state"] = {"page": halaman}
        _tulis_json_atomik(cfg.progress_path, progress)

        if batas_url is not None and len(urls) >= batas_url:
            urls = urls[:batas_url]
            break

        halaman += 1
    return urls

## 12) Scrape pipeline (incremental save)

In [ ]:
KOLOM_FINAL = ["url", "judul", "tanggal", "isi_berita", "Narasi", "Clean Narasi", "hoax", "summary", "source"]
_csv_lock = threading.Lock()


def _tulis_csv_append(path: Path, rows: List[Dict]):
    if not rows:
        return

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with _csv_lock:
        if not path.exists():
            tmp = path.with_suffix(path.suffix + ".tmp")
            with tmp.open("w", encoding="utf-8", newline="") as f:
                w = csv.DictWriter(f, fieldnames=KOLOM_FINAL)
                w.writeheader()
                for r in rows:
                    w.writerow({k: r.get(k, "") for k in KOLOM_FINAL})
            os.replace(tmp, path)
            return

        tmp_append = path.with_suffix(path.suffix + ".append.tmp")
        with tmp_append.open("w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=KOLOM_FINAL)
            for r in rows:
                w.writerow({k: r.get(k, "") for k in KOLOM_FINAL})

        with path.open("a", encoding="utf-8", newline="") as dst, tmp_append.open("r", encoding="utf-8") as src:
            dst.write(src.read())
            dst.flush()
            os.fsync(dst.fileno())

        try:
            tmp_append.unlink()
        except Exception:
            pass


def scrape_url(sumber: str, url: str, hoax_label: int) -> Optional[Dict]:
    if not robots.can_fetch(url):
        catat_failed(url, sumber, "robots", "RESTRICTED by robots")
        return None

    try:
        r = http_get(url)
        if r.status_code != 200:
            catat_failed(url, sumber, "http", f"status={r.status_code}")
            return None

        html = r.text
        if sumber == "cnn":
            data = parse_cnn(html, url)
            narasi = data.get("isi_berita", "")
        elif sumber == "detik":
            data = parse_detik(html, url)
            narasi = data.get("isi_berita", "")
        elif sumber == "kompas":
            data = parse_kompas(html, url)
            narasi = data.get("isi_berita", "")
        elif sumber == "turnbackhoaks":
            data = parse_turnbackhoaks(html, url)
            narasi = data.get("klaim") or data.get("isi_berita", "")
        else:
            catat_failed(url, sumber, "parse", "unknown source")
            return None

        tanggal_iso = data.get("tanggal")
        if not tanggal_iso:
            tanggal_iso = tanggal_dari_url(url)
        if not tanggal_iso or not dalam_rentang_2020(tanggal_iso):
            catat_failed(url, sumber, "date", f"invalid_or_out_of_range raw_src={data.get('tanggal_src')}")
            return None

        isi = data.get("isi_berita", "")
        judul = data.get("judul", "")
        narasi = narasi or isi or judul

        clean = bersihkan_teks_id(narasi)
        if not clean:
            clean = re.sub(r"\s+", " ", str(narasi).lower()).strip()
        summ = ringkas_ekstraktif(narasi, max_kalimat=2)
        if not summ:
            summ = ringkas_ekstraktif(isi or judul, max_kalimat=2)

        row = {
            "url": url,
            "judul": judul,
            "tanggal": tanggal_iso,
            "isi_berita": isi,
            "Narasi": narasi,
            "Clean Narasi": clean,
            "hoax": int(hoax_label),
            "summary": summ,
            "source": sumber,
        }

        if not row["url"] or not row["judul"] or not row["Narasi"] or not row["Clean Narasi"]:
            catat_failed(url, sumber, "validate", "missing critical fields")
            return None
        return row
    except Exception as e:
        catat_failed(url, sumber, "exception", str(e))
        return None


def scrape_sumber(
    sumber: str,
    urls: List[str],
    hoax_label: int,
    output_csv: Path,
    limit: Optional[int] = None,
):
    urls = list(dict.fromkeys(urls))
    total_input = len(urls)

    if limit is not None:
        urls = urls[:limit]

    urls = done_store.filter_not_done(sumber, urls)
    total_proses = len(urls)
    if not urls:
        logger.info("[%s] Tidak ada URL baru untuk diproses.", sumber)
        mark_meta_progress(
            progress,
            sumber,
            last_run=time.time(),
            last_total_urls_input=total_input,
            last_total_urls_to_process=0,
            last_success=0,
            last_failed=0,
        )
        return

    logger.info("[%s] Mulai scrape: %d URL", sumber, len(urls))
    from concurrent.futures import ThreadPoolExecutor, as_completed

    batch_rows: List[Dict] = []
    success_urls_batch: List[str] = []
    n_sukses = 0
    n_gagal = 0

    with ThreadPoolExecutor(max_workers=cfg.max_workers_total) as ex:
        futures = {ex.submit(scrape_url, sumber, u, hoax_label): u for u in urls}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"scrape:{sumber}"):
            u = futures[fut]
            row = fut.result()
            if row:
                n_sukses += 1
                batch_rows.append(row)
                success_urls_batch.append(u)
            else:
                n_gagal += 1

            if len(batch_rows) >= 200:
                _tulis_csv_append(output_csv, batch_rows)
                # penting: done-store hanya URL sukses
                done_store.add_many(sumber, success_urls_batch)
                batch_rows.clear()
                success_urls_batch.clear()

    if batch_rows:
        _tulis_csv_append(output_csv, batch_rows)
    if success_urls_batch:
        done_store.add_many(sumber, success_urls_batch)

    mark_meta_progress(
        progress,
        sumber,
        last_run=time.time(),
        last_total_urls_input=total_input,
        last_total_urls_to_process=total_proses,
        last_success=n_sukses,
        last_failed=n_gagal,
    )
    logger.info("[%s] Selesai. success=%d failed=%d Output: %s", sumber, n_sukses, n_gagal, output_csv)

## 13) Jalankan discovery + scrape (DEFAULT: full-run tanpa batas)
Mode smoke-test: set `cfg.mode_smoke_test=True` sebelum menjalankan cell ini.

In [ ]:
# 13.1 Load robots untuk domain target (wajib sebelum discovery/scrape)
domain_targets = [
    "https://www.cnnindonesia.com",
    "https://www.detik.com",
    "https://news.detik.com",
    "https://www.kompas.com",
    "https://turnbackhoax.id",
    "https://www1.turnbackhoax.id",
]

rows = []
for d in domain_targets:
    ok, note = robots.load(d)
    rows.append({"domain": d, "robots_loaded": ok, "note": note})

display(pd.DataFrame(rows))

# Pastikan URL sukses existing sudah masuk done-store agar rerun tidak duplikasi
seed_done_store_dari_csv()
print("done_success_urls by source:", done_store.count_by_source())

In [ ]:
# 13.2 Discovery
urls_cnn = temukan_url_cnn()
urls_detik = temukan_url_detik()
urls_kompas = temukan_url_kompas()
urls_tbh = temukan_url_turnbackhoaks()

print("Discovery counts:", {
    "cnn": len(urls_cnn),
    "detik": len(urls_detik),
    "kompas": len(urls_kompas),
    "turnbackhoaks": len(urls_tbh),
})

In [ ]:
# 13.3 Scrape (full-run default; smoke-test akan membatasi)
out_cnn = cfg.output_dir / "data_nonhoaks_cnn.csv"
out_detik = cfg.output_dir / "data_nonhoaks_detik.csv"
out_kompas = cfg.output_dir / "data_nonhoaks_kompas.csv"
out_tbh = cfg.output_dir / "data_hoaks_turnbackhoaks.csv"

paths_out = {
    "turnbackhoaks": out_tbh,
    "cnn": out_cnn,
    "detik": out_detik,
    "kompas": out_kompas,
}

limit_scrape = cfg.smoke_limit_scrape_per_sumber if cfg.mode_smoke_test else None

scrape_sumber("turnbackhoaks", urls_tbh, hoax_label=1, output_csv=out_tbh, limit=limit_scrape)
scrape_sumber("cnn", urls_cnn, hoax_label=0, output_csv=out_cnn, limit=limit_scrape)
scrape_sumber("detik", urls_detik, hoax_label=0, output_csv=out_detik, limit=limit_scrape)
scrape_sumber("kompas", urls_kompas, hoax_label=0, output_csv=out_kompas, limit=limit_scrape)

## 14) Gabungkan dataset (opsional, default ON)

In [ ]:
gabung_path = cfg.output_dir / "dataset_final_gabungan.csv"


def gabungkan_csv_ke_satu(out_path: Path, input_paths: Dict[str, Path]):
    # Tulis ke file sementara lalu rename atomik, tanpa menghapus file lama secara langsung.
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = out_path.with_suffix(out_path.suffix + ".tmp")

    if tmp_path.exists():
        try:
            tmp_path.unlink()
        except Exception:
            pass

    for src, pth in input_paths.items():
        pth = Path(pth)
        if not pth.exists():
            continue
        for chunk in pd.read_csv(pth, chunksize=200000):
            chunk = chunk[KOLOM_FINAL]
            chunk.to_csv(tmp_path, mode="a", index=False, header=not tmp_path.exists())

    if tmp_path.exists():
        os.replace(tmp_path, out_path)


if cfg.gabungkan_output:
    gabungkan_csv_ke_satu(gabung_path, paths_out)
    logger.info("Gabungan selesai: %s", gabung_path)
    print("Gabungan:", gabung_path)

## 15) Troubleshooting

### 403/429
1. Turunkan `max_workers_total` dan naikkan `min_jeda_per_domain_detik`.
2. Pastikan user-agent akademik tetap sama.
3. Jika robots melarang, hentikan scraping pada domain tersebut.

### Selector berubah
1. Update selector pada parser `parse_cnn`, `parse_detik`, `parse_kompas`, `parse_turnbackhoaks`.
2. Pertahankan fallback `trafilatura`.
3. Jalankan smoke test sebelum full run.

### Cache corrupt
1. Hapus file sitemap cache yang rusak di `scrape_output/cache/`.
2. Jalankan discovery ulang.

### Resume aman
1. `done_success_urls` hanya menandai URL sukses.
2. Rerun tidak menduplikasi URL sukses karena seed dari CSV existing + sqlite.

## 16) QA Cell F1 - Audit `scrape_output`
Ringkasan untuk tiap CSV:
- rows, missing rate kolom penting, dup URL, distribusi label, contoh 3 URL
Ringkasan artefak lain:
- failed_urls top reason
- progress_scrape status terakhir
- done_success_urls sqlite per sumber

In [ ]:
KOLOM_MINIMAL = ["url", "judul", "tanggal", "isi_berita", "Narasi", "Clean Narasi", "hoax", "summary", "source"]


def audit_csv_lengkap(path: Path) -> Dict:
    path = Path(path)
    if not path.exists():
        return {
            "rows": 0,
            "missing_required_cols": KOLOM_MINIMAL,
            "missing_rate": {},
            "dup_url": 0,
            "label_dist": {},
            "samples": [],
        }

    header = list(pd.read_csv(path, nrows=0).columns)
    missing_cols = [c for c in KOLOM_MINIMAL if c not in header]

    total = 0
    miss_counter = Counter()
    label_counter = Counter()
    sample_rows = []

    tmp_db = cfg.cache_dir / f"audit_urls_{hashlib.sha1(str(path).encode('utf-8')).hexdigest()}.sqlite"
    if tmp_db.exists():
        try:
            tmp_db.unlink()
        except Exception:
            pass

    con = sqlite3.connect(tmp_db)
    try:
        con.execute("PRAGMA journal_mode=WAL;")
        con.execute("CREATE TABLE IF NOT EXISTS urls (url TEXT PRIMARY KEY)")
        con.commit()

        for chunk in pd.read_csv(path, chunksize=150000, dtype=str, keep_default_na=False):
            n = len(chunk)
            total += n

            for col in KOLOM_MINIMAL:
                if col in chunk.columns:
                    s = chunk[col].fillna("").astype(str).str.strip()
                    miss = (s == "") | s.str.lower().isin(["nan", "none", "null"])
                    miss_counter[col] += int(miss.sum())

            if "hoax" in chunk.columns:
                label_counter.update(chunk["hoax"].fillna("").astype(str).str.strip().tolist())

            if "url" in chunk.columns:
                urls = chunk["url"].fillna("").astype(str).str.strip().tolist()
                con.executemany("INSERT OR IGNORE INTO urls(url) VALUES (?)", [(u,) for u in urls if u])
                con.commit()

            if len(sample_rows) < 3:
                cols = [c for c in ["url", "judul", "tanggal", "hoax", "source"] if c in chunk.columns]
                sample_rows.extend(chunk[cols].head(3 - len(sample_rows)).to_dict("records"))

        uniq = con.execute("SELECT COUNT(*) FROM urls").fetchone()[0]
    finally:
        con.close()

    try:
        tmp_db.unlink()
    except Exception:
        pass

    dup = max(0, total - int(uniq))
    miss_rate = {
        col: (round((miss_counter[col] / total) * 100, 2) if total and col in header else None)
        for col in KOLOM_MINIMAL
    }

    return {
        "rows": total,
        "missing_required_cols": missing_cols,
        "missing_rate": miss_rate,
        "dup_url": dup,
        "label_dist": dict(label_counter),
        "samples": sample_rows,
    }


paths_out = {
    "turnbackhoaks": cfg.output_dir / "data_hoaks_turnbackhoaks.csv",
    "cnn": cfg.output_dir / "data_nonhoaks_cnn.csv",
    "detik": cfg.output_dir / "data_nonhoaks_detik.csv",
    "kompas": cfg.output_dir / "data_nonhoaks_kompas.csv",
}

audit = {src: audit_csv_lengkap(p) for src, p in paths_out.items()}

ringkas_rows = []
for src, obj in audit.items():
    ringkas_rows.append({
        "source": src,
        "rows": obj["rows"],
        "dup_url": obj["dup_url"],
        "missing_required_cols": ", ".join(obj["missing_required_cols"]) if obj["missing_required_cols"] else "-",
        "label_dist": obj["label_dist"],
        "sample_urls": [r.get("url", "") for r in obj["samples"]],
    })

print("Audit CSV per sumber:")
display(pd.DataFrame(ringkas_rows))

print("\nMissing rate kolom penting (%):")
for src, obj in audit.items():
    print(f"- {src}: {obj['missing_rate']}")

print("\nContoh baris ringkas (maks 3 per sumber):")
for src, obj in audit.items():
    print(f"[{src}]")
    for r in obj["samples"]:
        judul = str(r.get("judul", ""))
        judul = judul[:90] + ("..." if len(judul) > 90 else "")
        print({"url": r.get("url", ""), "judul": judul, "tanggal": r.get("tanggal", ""), "hoax": r.get("hoax", "")})

# failed_urls summary
failed_summary = {"total": 0, "top_reasons": [], "top_stages": [], "top_sources": []}
if cfg.failed_path.exists():
    reason_ctr = Counter()
    stage_ctr = Counter()
    source_ctr = Counter()

    with cfg.failed_path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            failed_summary["total"] += 1
            try:
                obj = json.loads(line)
            except Exception:
                reason_ctr["<invalid_json_line>"] += 1
                continue
            reason_ctr[obj.get("reason", "<none>")] += 1
            stage_ctr[obj.get("stage", "<none>")] += 1
            source_ctr[obj.get("source", "<none>")] += 1

    failed_summary["top_reasons"] = reason_ctr.most_common(5)
    failed_summary["top_stages"] = stage_ctr.most_common(5)
    failed_summary["top_sources"] = source_ctr.most_common(5)

print("\nRingkasan failed_urls:", failed_summary)

# progress summary
progress_rows = []
if cfg.progress_path.exists():
    pjson = json.loads(cfg.progress_path.read_text(encoding="utf-8"))
    for src, d in pjson.get("sumber", {}).items():
        meta = d.get("meta", {})
        progress_rows.append({
            "source": src,
            "last_total_urls": meta.get("last_total_urls"),
            "last_total_urls_input": meta.get("last_total_urls_input"),
            "last_total_urls_to_process": meta.get("last_total_urls_to_process"),
            "last_success": meta.get("last_success"),
            "last_failed": meta.get("last_failed"),
            "last_run": pd.to_datetime(meta.get("last_run"), unit="s").strftime("%Y-%m-%d %H:%M:%S") if meta.get("last_run") else None,
            "discovery_page": d.get("discovery_state", {}).get("page"),
        })

print("\nProgress scrape status:")
display(pd.DataFrame(progress_rows))
print("done_success_urls by source:", done_store.count_by_source())

# log summary
log_path = cfg.logs_dir / "scrape.log"
if log_path.exists():
    lines = [ln for ln in log_path.read_text(encoding="utf-8", errors="replace").splitlines() if ln.strip()]
    warn_err_ctr = Counter()
    for ln in lines:
        if "| WARNING |" in ln or "| ERROR |" in ln:
            warn_err_ctr[ln.split("|", maxsplit=2)[-1].strip()] += 1
    print("Top warning/error di scrape.log:", warn_err_ctr.most_common(5))

# kesimpulan audit
kesenjangan = []
for src, obj in audit.items():
    if obj["missing_required_cols"]:
        kesenjangan.append(f"{src}: kolom kurang {obj['missing_required_cols']}")

if failed_summary["total"] > 0:
    top_reason = failed_summary["top_reasons"][0][0] if failed_summary["top_reasons"] else "-"
    kesenjangan.append(f"failed_urls masih ada ({failed_summary['total']}), top reason={top_reason}")

if kesenjangan:
    print("\nKesimpulan: BELUM SEPENUHNYA SIAP, perlu perbaikan:")
    for k in kesenjangan:
        print("-", k)
else:
    print("\nKesimpulan: scrape_output sudah sesuai kebutuhan dataset training.")

## 17) QA Cell F2 - Kesesuaian untuk Training
Pemeriksaan:
- kolom yang dipakai downstream tersedia
- label global memuat {0,1}
- summary mayoritas tidak kosong
- cross-check ringkas dengan build_dataset.py dan data/processed

In [ ]:
kolom_training_wajib = ["url", "judul", "tanggal", "isi_berita", "Narasi", "Clean Narasi", "summary", "hoax", "source"]

status_ok = True
alasan = []
label_global = set()
summary_ratio = {}

paths_out = {
    "turnbackhoaks": cfg.output_dir / "data_hoaks_turnbackhoaks.csv",
    "cnn": cfg.output_dir / "data_nonhoaks_cnn.csv",
    "detik": cfg.output_dir / "data_nonhoaks_detik.csv",
    "kompas": cfg.output_dir / "data_nonhoaks_kompas.csv",
}

for src, p in paths_out.items():
    p = Path(p)
    if not p.exists():
        status_ok = False
        alasan.append(f"{src}: file tidak ditemukan")
        continue

    cols = list(pd.read_csv(p, nrows=0).columns)
    miss = [c for c in kolom_training_wajib if c not in cols]
    if miss:
        status_ok = False
        alasan.append(f"{src}: kolom wajib tidak lengkap {miss}")

    total = 0
    nonempty_summary = 0
    for chunk in pd.read_csv(p, chunksize=150000, dtype=str, keep_default_na=False):
        total += len(chunk)
        if "hoax" in chunk.columns:
            vals = chunk["hoax"].fillna("").astype(str).str.strip().tolist()
            label_global.update([v for v in vals if v])
        if "summary" in chunk.columns:
            s = chunk["summary"].fillna("").astype(str).str.strip()
            nonempty_summary += int((s != "").sum())

    ratio = (nonempty_summary / total) if total else 0.0
    summary_ratio[src] = round(ratio, 4)
    if ratio < 0.5:
        status_ok = False
        alasan.append(f"{src}: summary non-empty ratio rendah ({ratio:.2%})")

if not {"0", "1"}.issubset(label_global):
    status_ok = False
    alasan.append(f"Label global belum lengkap, ditemukan={sorted(label_global)}")

print("summary_non_empty_ratio:", summary_ratio)
print("label_global:", sorted(label_global))

builder = cfg.project_root / "scripts" / "build_dataset.py"
if builder.exists():
    txt = builder.read_text(encoding="utf-8", errors="replace")
    print("build_dataset.py ditemukan:", builder)
    print("contains REQUIRED_COLUMNS:", "REQUIRED_COLUMNS" in txt)
else:
    status_ok = False
    alasan.append("MISSING: scripts/build_dataset.py")

summary_path = cfg.project_root / "data" / "processed" / "summary.json"
leak_path = cfg.project_root / "data" / "processed" / "leakage_audit.json"
if summary_path.exists():
    obj = json.loads(summary_path.read_text(encoding="utf-8"))
    print("processed rows_total:", obj.get("rows_total"))
    print("processed leakage pass:", obj.get("leakage", {}).get("pass"))
if leak_path.exists():
    obj = json.loads(leak_path.read_text(encoding="utf-8"))
    print("leakage_audit pass:", obj.get("pass"), "max_gap:", obj.get("max_gap"))



deteksi_nb = cfg.project_root / "Deteksi_Hoax_NER_Optimized.ipynb"
if deteksi_nb.exists():
    txt_nb = deteksi_nb.read_text(encoding="utf-8", errors="replace")
    print("Deteksi_Hoax_NER_Optimized.ipynb ditemukan:", deteksi_nb)
    print("mengandung referensi build_dataset.py:", "build_dataset.py" in txt_nb)
    print("mengandung train.csv/val.csv/test.csv:", all(k in txt_nb for k in ["train.csv", "val.csv", "test.csv"]))
else:
    status_ok = False
    alasan.append("MISSING: Deteksi_Hoax_NER_Optimized.ipynb")


if status_ok:
    print("\nKesesuaian untuk Training: OK")
else:
    print("\nKesesuaian untuk Training: NOT OK")
    for a in alasan:
        print("-", a)